# Train two-tower + cross-attention classifier - StereoQueerEval Task A

Binary stereotype detection (`yes` / `no`) with `src/train_xattn.py`.

The comment is the **query**; the video (title + description) is the
**key/value memory** - exactly the cross-attention of a Transformer decoder,
without any generation. Pool the comment side and read a 2-way logit.

Design here: MiniLM encoder + **stacked cross-attention blocks**, with an
optional **two-phase** schedule (keep the encoder frozen first, then unfreeze the
top blocks once the head has settled).

- Encoder: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384-d)
- Video budget 256 tokens: all videos fit (description max measured 216)
- Same video-level split / labels / metrics as `train.py`
- Frozen phase => the video K/V is deterministic and cacheable

Works on **Kaggle** and Colab.


In [ ]:
import os, subprocess

if not os.path.isdir('Try'):
    subprocess.run('git clone https://github.com/AmnO-O/Try.git', shell=True, check=True)
os.chdir('Try')
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print('cwd:', os.getcwd())
print('src has train_xattn.py:', os.path.exists('src/train_xattn.py'))


## 1. Locate the data

The TSVs are not committed. This cell finds them automatically (Kaggle input,
`/content`, cwd, `./LGBT`) and exports `LGBT_DATA_DIR`.


In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.environ['KAGGLE_INPUT_PATH'])
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return hits
    return []

hits = _first(candidates)
if not hits:
    raise RuntimeError('StereoQueerEval_*_training.tsv not found - mount the dataset first')
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs(os.environ['LGBT_WEIGHTS_DIR'], exist_ok=True)
print('data dir :', os.environ['LGBT_DATA_DIR'])
print('weights  :', os.environ['LGBT_WEIGHTS_DIR'])
print('files    :', [os.path.basename(p) for p in hits])
assert len(hits) >= 1


## 2. Sanity check the split

Row counts, label balance and the video-grouped train/val split. Only **EN / IT /
NL** are loaded - the FA files stay held out (if `StereoQueerEval_FA_*` is present
in the data dir, loading *all* languages would leak it into train and val).
Defaults are `val-fraction=0.1`, `split-seed=42` - the same ones the cells below use.


In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod
from config import MAX_LEN

LANGS = ['EN', 'IT', 'NL']
df = data_mod.load_frame(languages=LANGS, max_len=MAX_LEN)
print('rows:', len(df), '| by lang:', df['lang'].value_counts().to_dict())
tr, va = data_mod.split_by_video(df, 0.1, 42)
print('train/val:', len(tr), '/', len(va))
print('video overlap:', len(set(tr['yt_title']) & set(va['yt_title'])))


## 3. Train the cross-attention model

MiniLM encoder + **3 cross-attention blocks** (4 heads), light regularization
(dropout 0.1, weight decay 0.01, head LR 1e-3), and the top **2 encoder blocks
unfrozen** - the best 3-seed mean so far (macro-F1 0.7097 +/- 0.028 over seeds
40/41/42).

The encoder runs in `eval()` / under `no_grad` while frozen, so during the frozen
phase its features (and the video K/V) are deterministic. Section 6 screens where
the **title** goes: it can be *moved* into the query (`--query-fields title,comment`)
or *added* there while staying in the K/V stream (`--keep-kv-title`).


In [ ]:
!python src/train_xattn.py --languages EN IT NL --epochs 20 --batch-size 32 --max-len 160 --video-max-len 256 --unfreeze-layers 2 --heads 4 --layers 3 --dropout 0.1 --weight-decay 0.01 --head-lr 1e-3


## 4. Curves

Reads `weights/best_xattn_stereotype_history.csv` written by the cell above.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

h = pd.read_csv('weights/best_xattn_stereotype_history.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(h['epoch'], h['train_loss'], '--', label='train')
ax[0].plot(h['epoch'], h['val_loss'], '-', label='val')
ax[1].plot(h['epoch'], h['val_macro_f1'], '-o', ms=3, label='val macro-F1')
ax[0].set_title('loss'); ax[1].set_title('validation macro-F1')
for a in ax:
    a.grid(alpha=0.3); a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.show()

best = h.loc[h['val_macro_f1'].idxmax()]
print(f"best epoch {int(best['epoch'])} | macro_f1 {best['val_macro_f1']:.4f} | acc {best['val_acc']:.4f}")


## 5. Confusion matrix + per-language (best checkpoint)

Reloads `weights/best_xattn_stereotype.pt` with the **same token budgets used in
training** (`max-len 160`, `video-max-len 256`), then scores the same
video-grouped validation split (EN/IT/NL only).


In [ ]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

from model_xattn import build_model, DEFAULT_BACKBONE
from model import load_tokenizer
from train import resolve_device
import train_xattn as tx

dev = resolve_device('cuda' if torch.cuda.is_available() else 'cpu')
tok = load_tokenizer(DEFAULT_BACKBONE)
net = build_model(DEFAULT_BACKBONE, unfreeze_layers=2, heads=4, layers=3)
net.load_state_dict(torch.load('weights/best_xattn_stereotype.pt', map_location=dev))
net.to(dev).eval()

va = va.reset_index(drop=True)
out = tx.collect(net, tx.make_loader(va, tok, 32, 160, 256), dev)
m = tx.metrics_from(out['preds'], out['labels'])
print({k: round(v, 4) for k, v in m.items() if k != 'loss'})

cm = confusion_matrix(out['labels'], out['preds'], labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['no', 'yes'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['no', 'yes'])
ax.set_title('xattn (best checkpoint)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

for lang in sorted(va['lang'].unique()):
    sub = va[va['lang'] == lang].reset_index(drop=True)
    o = tx.collect(net, tx.make_loader(sub, tok, 32, 160, 256), dev)
    mm = tx.metrics_from(o['preds'], o['labels'])
    print(f"{lang}: acc {mm['acc']:.4f} | macro_f1 {mm['macro_f1']:.4f}")


## 6. Screen: title placement (seed 42)

Does giving the encoder the title in-sequence help, on top of the K/V paths?

- `u2_l3`       - query = `comment`, K/V = `title [SEP] description` (control)
- `u2_l3_tq`    - query = `title [SEP] comment`, K/V = `description` (title *moved*)
- `u2_l3_tq_kv` - query = `title [SEP] comment`, K/V = `title [SEP] description` (title *added*)

`tq` drops the title from the video stream; `tq_kv` keeps it, so this pair
separates "title in-sequence" from "title removed from K/V".

Seed 42 is our easiest split and the unfrozen config has ~1-1.5 pt run-to-run
jitter, so only a difference above ~2 pt is readable at n=1.


In [ ]:
import subprocess
import pandas as pd

SEEDS = (42,)   # widen to (40, 41, 42) to confirm a candidate before trusting it
BASE = ('--languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        '--max-len 160 --video-max-len 256 --dropout 0.1 --weight-decay 0.01 --head-lr 1e-3')
CONFIGS = {
    'u2_l3':       '--unfreeze-layers 2 --layers 3',
    'u2_l3_tq':    '--unfreeze-layers 2 --layers 3 --query-fields title,comment',
    'u2_l3_tq_kv': '--unfreeze-layers 2 --layers 3 --query-fields title,comment --keep-kv-title',
}

rows = []
for name, flags in CONFIGS.items():
    for s in SEEDS:
        out = f'weights/xattn_{name}_s{s}.pt'
        cmd = f'python src/train_xattn.py --split-seed {s} --seed {s} {BASE} {flags} --out {out}'
        print('RUN:', cmd)
        subprocess.run(cmd, shell=True, check=True)
        h = pd.read_csv(out.replace('.pt', '_history.csv'))
        rows.append({'config': name, 'seed': s,
                     'val_macro_f1': round(float(h['val_macro_f1'].max()), 4)})

df = pd.DataFrame(rows)
df.to_csv('weights/xattn_confirm.csv', index=False)
summary = (df.groupby('config')['val_macro_f1']
             .agg(mean='mean', std='std', n='count').round(4)
             .sort_values('mean', ascending=False))
print(summary.to_string())
print()
print(df.pivot(index='config', columns='seed', values='val_macro_f1').to_string())


## 7. Compare to the prompt baseline (`src/train.py`)

Same seeds and split, but the mmBERT prompt classifier instead of the two-tower
model. Reference best so far: **0.7158** (batch 32, no extras). Averages the
baseline over the same seeds as section 6 (re-runs it if
`weights/xattn_confirm.csv` is present).


In [ ]:
import os
import subprocess
import pandas as pd

SEEDS = (40, 41, 42)
BASE = ('--languages EN IT NL --epochs 10 --batch-size 32 --max-len 320 '
        '--unfreeze-layers 2 --readout-init random')

rows = []
for s in SEEDS:
    out = f'weights/baseline_s{s}.pt'
    cmd = f'python src/train.py --split-seed {s} --seed {s} {BASE} --out {out}'
    print('RUN:', cmd)
    subprocess.run(cmd, shell=True, check=True)
    h = pd.read_csv(out.replace('.pt', '_history.csv'))
    rows.append({'config': 'prompt_baseline', 'seed': s,
                 'val_macro_f1': round(float(h['val_macro_f1'].max()), 4)})

df = pd.DataFrame(rows)
if os.path.exists('weights/xattn_confirm.csv'):
    df = pd.concat([df, pd.read_csv('weights/xattn_confirm.csv')], ignore_index=True)

summary = (df.groupby('config')['val_macro_f1']
             .agg(mean='mean', std='std', n='count').round(4)
             .sort_values('mean', ascending=False))
print(summary.to_string())


### Without a notebook

```bash
git clone https://github.com/AmnO-O/Try.git && cd Try
pip install -r requirements.txt
export LGBT_DATA_DIR=/path/to/tsvs            # or put them in ./LGBT/

# best single-phase config: MiniLM + 3 cross-attn blocks, top 2 blocks trainable
python src/train_xattn.py --languages EN IT NL --epochs 20 --batch-size 32 \
    --max-len 160 --video-max-len 256 --unfreeze-layers 2 --layers 3

# two-phase: encoder frozen for 5 epochs, then unfreeze the top 2 blocks
python src/train_xattn.py --languages EN IT NL --freeze-epochs 5 \
    --unfreeze-layers 2 --layers 3

# move the title into the query: comment [SEP] title, K/V = description
python src/train_xattn.py --languages EN IT NL --query-fields comment,title \
    --unfreeze-layers 2 --layers 3

# same fields, title first (order follows the flag)
python src/train_xattn.py --languages EN IT NL --query-fields title,comment \
    --unfreeze-layers 2 --layers 3

# title in the query AND kept in K/V (query `title | comment`, K/V `title | description`)
python src/train_xattn.py --languages EN IT NL --query-fields title,comment \
    --keep-kv-title --unfreeze-layers 2 --layers 3

# prompt baseline on the same split (EN/IT/NL)
python src/train.py --languages EN IT NL --epochs 10 --batch-size 32 \
    --max-len 320 --unfreeze-layers 2 --readout-init random

# swap the encoder / deepen the cross-attention
python src/train_xattn.py --backbone jhu-clsp/mmbert-base --layers 4

# augmented train split (built with src/augment_roundtrip.py)
python src/train_xattn.py --languages EN IT NL \
    --train-extras aug_rt_IT_to_EN_training.tsv,aug_rt_NL_to_EN_training.tsv
```

Checkpoints and history CSVs land in `weights/`.
